In [1]:
import os, sys, math
# Runtime only: put the v6 repo dir on sys.path so `import modules` works from
# notebooks/. Editor go-to-definition comes from python.analysis.extraPaths.
_V6 = os.path.abspath(".." if os.path.isdir("../modules") else ".")
if _V6 not in sys.path:
    sys.path.insert(0, _V6)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import modules  # noqa: F401 — sys.path injection for v3 + v4

# A kernel keeps every module it has already imported in sys.modules, so editing a
# pipeline module on disk mid-session does NOT take effect on re-run — and pulling a
# newly added name out of it fails with a confusing
#   ImportError: cannot import name '<new_fn>' from 'modules.<mod>'
# even though the function is right there in the file. Reload the pipeline modules
# (dependencies first) so re-running this cell always tracks the current source.
import importlib
for _m in ("modules.constants", "modules.geometry",
           "modules.layouts.strategies", "modules.data",
           "modules.optimize"):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])
# NE (East, North) pipeline modules — the ones Step 1 actually uses. The old
# (North, Up) `fnn_surrogate` / `detector_strategies` / `SurfaceEastMap` trio this
# notebook used no longer matches the builder.
from modules.data     import compute_labels_batch, place_clouds_enu
from modules.layouts.strategies import _STRATEGIES, _STRATEGY_FNS
from modules.geometry    import SurfaceUpMap, load_tr_mountain
from modules.optimize             import LAYOUT_THRESHOLD
from modules.constants import (
    GEOMETRY_PATH_RESOLVED, GEOMETRY_GROUP, DET_KEY,
    EAST_ENTRY, LAYER_EAST_DX, N_PLANES, N_DETECTORS, SIGMA_SPATIAL,
    DUAL_SHOWER_CACHE_PATH, DUAL_POSITIONS_PATH, DUAL_SPECIES_IDS_PATH,
    LOG_E_MIN, LOG_E_MAX,
)
import showerdata

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device={DEVICE}  LAYOUT_THRESHOLD={LAYOUT_THRESHOLD}  SIGMA_SPATIAL={SIGMA_SPATIAL}')

mountain = load_tr_mountain(GEOMETRY_PATH_RESOLVED, GEOMETRY_GROUP, DET_KEY,
                            east_entry=EAST_ENTRY, layer_east_dx=LAYER_EAST_DX,
                            n_planes=N_PLANES)
surface  = SurfaceUpMap.from_mountain(mountain, grid_h=256, grid_w=256).to(DEVICE)
_mE, _mN, _mU = (mountain.centroids_ENU[:, i] for i in range(3))
print(f'mountain  East=[{_mE.min():.0f}, {_mE.max():.0f}]  North=[{_mN.min():.0f}, {_mN.max():.0f}]'
      f'  Up=[{_mU.min():.0f}, {_mU.max():.0f}]')

# ── Real tau corpus, STREAMED ────────────────────────────────────────────────
# cashed_showers_tau_dual.pt is ~22 GB (electron block then muon block), far too
# large to load whole. Take a contiguous slice of the ELECTRON block and stream
# only those rows. Row i of the sidecars is aligned with corpus row i.
N_SHOWERS_NB = 300            # showers pulled into the notebook
positions_all = torch.load(DUAL_POSITIONS_PATH)                 # (M,3) ENU East,North,Up
species_all   = torch.load(DUAL_SPECIES_IDS_PATH)               # (M,) idx into SPECIES_NAMES=('electron','muon','photon')
n_pairs = int((species_all == 0).sum())
sl = slice(0, min(N_SHOWERS_NB, n_pairs))                       # electron block
sub = showerdata.load(DUAL_SHOWER_CACHE_PATH, start=sl.start, stop=sl.stop)
points = torch.as_tensor(sub.points,     dtype=torch.float32)   # (S,P,5) NATIVE frame
dirs   = torch.as_tensor(sub.directions, dtype=torch.float32)
energs = torch.as_tensor(sub.energies,   dtype=torch.float32).reshape(-1)
pdg    = torch.as_tensor(sub.pdg,        dtype=torch.long)
del sub
positions = positions_all[sl.start:sl.stop].float()             # (S,3) real decay vertices
print(f'corpus: {n_pairs} pairs; loaded {tuple(points.shape)} electron rows '
      f'(E {float(energs.min()):.2e}..{float(energs.max()):.2e} GeV)')

# ── Place at the REAL decay vertices (the Step-1 transform) ──────────────────
# Placement is always the real ENU decay vertex from tau_wholesky.jl — the only
# positioning the pipeline has. This calls the pipeline's own `place_clouds_enu`,
# the same function 01_build_dataset_northeast.py reaches through
# build_training_pairs, so these trigger counts are exactly the ones Step 1 produces.
bad = ~torch.isfinite(points).all(dim=-1)
if int(bad.sum()):
    points[bad] = 0.0
    print(f'[sanitize] zeroed {int(bad.sum())} non-finite points')
dirs_u = dirs / dirs.norm(dim=1, keepdim=True).clamp(min=1e-12)   # unit travel dirs
points = place_clouds_enu(points, positions, dirs_u,
                          east_entry=EAST_ENTRY, layer_east_dx=LAYER_EAST_DX)
print(f'[place] {points.shape[0]} clouds at real decay vertices '
      f'(east_entry={EAST_ENTRY:g}, dx={LAYER_EAST_DX:g})')

device=cpu  LAYOUT_THRESHOLD=1.0  SIGMA_SPATIAL=50.0
mountain  East=[-498, 777]  North=[-956, 716]  Up=[2748, 3712]
corpus: 572164 pairs; loaded (300, 25088, 5) electron rows (E 1.07e+05..9.95e+06 GeV)
[place] 300 clouds at real decay vertices (east_entry=1500, dx=500)


## Pick a diverse set of showers

We want primaries spanning energy and zenith. Screen on a default-grid layout first and
keep the ones that actually deposit — with the placement fixed this now retains ~90% of
the sample rather than ~2%.

In [2]:
# Screen the loaded showers on one default grid layout.
#
# THRESHOLDS: LAYOUT_THRESHOLD is the 04 reconstructability cut (imported, so this
# cell tracks it). Before the place_clouds_enu frame fix only ~2% of showers cleared
# it (the clouds were placed kilometres off their own axis); it is now ~90%, in line
# with the ~91% whose axis passes within SIGMA_SPATIAL of a detector. The nonzero
# count and summed response are still recorded so the screen degrades gracefully on
# a thin sample.
rng_scan = np.random.default_rng(42)
fn_grid  = _STRATEGY_FNS['layout_grid']
e_scan, n_scan = fn_grid(mountain, n_det=N_DETECTORS, rng=rng_scan)   # (East, North)
e_scan = e_scan.float().to(DEVICE); n_scan = n_scan.float().to(DEVICE)

scan_records = []
for s in range(points.shape[0]):                      # every loaded shower
    E, _ = compute_labels_batch(points[s:s+1].to(DEVICE), e_scan, n_scan, surface)
    E0 = E[0]
    # theta/phi in the pipeline's own convention, so they are the quantities
    # U_theta / U_phi are scored against: `dirs` are ENU unit vectors
    # (sin0 cos1, sin0 sin1, cos0), i.e. col 0 = East = dir_x, col 1 = North = dir_y,
    # col 2 = Up = dir_z — exactly what opt_core.primary_to_physical_labels reads.
    # phi is the East->North azimuth, wrapped to [0, 360).
    theta = float(torch.arccos(dirs_u[s, 2].clamp(-1.0, 1.0)).item())
    phi   = float(torch.atan2(dirs_u[s, 1], dirs_u[s, 0]).item())
    scan_records.append(dict(
        shower=s, E=float(energs[s].item()), theta_deg=math.degrees(theta),
        phi_deg=math.degrees(phi) % 360.0,
        n_trig=int((E0 > LAYOUT_THRESHOLD).sum().item()),   # above the 04 cut
        n_nonzero=int((E0 > 0).sum().item()),               # any response at all
        e_sum=float(E0.sum().item()),
        e_max=float(E0.max().item()),
    ))
df_scan = pd.DataFrame(scan_records)
n = len(df_scan)
print(f"scanned {n} showers on the default grid")
print(f"  any detector > LAYOUT_THRESHOLD={LAYOUT_THRESHOLD}: "
      f"{int((df_scan.n_trig > 0).sum())}/{n} ({100*(df_scan.n_trig > 0).mean():.1f}%)")
print(f"  any nonzero detector response          : "
      f"{int((df_scan.n_nonzero > 0).sum())}/{n} ({100*(df_scan.n_nonzero > 0).mean():.1f}%)")
print(f"  max single-detector response over all  : {df_scan.e_max.max():.4g}")
df_scan[['E', 'theta_deg', 'phi_deg', 'n_trig', 'n_nonzero', 'e_sum']].describe()

scanned 300 showers on the default grid
  any detector > LAYOUT_THRESHOLD=1.0: 246/300 (82.0%)
  any nonzero detector response          : 281/300 (93.7%)
  max single-detector response over all  : 6.98e+05


,E,theta_deg,phi_deg,n_trig,n_nonzero,e_sum
count,3.000000e+02,300.000000,300.000000,300.000000,300.000000,3.000000e+02
mean,1.237861e+06,84.754198,121.040213,26.853333,79.180000,3.095364e+05
std,1.532674e+06,11.955181,59.314082,19.314987,29.272693,5.784604e+05
min,1.066225e+05,58.896940,24.999583,0.000000,0.000000,0.000000e+00
25%,3.393372e+05,76.900209,81.764698,7.750000,74.000000,1.021514e+02
50%,6.175660e+05,84.846649,113.632945,29.000000,93.000000,5.550397e+04
75%,1.393007e+06,91.388269,144.498755,40.000000,100.000000,3.884317e+05
max,9.948421e+06,119.394862,358.705356,79.000000,100.000000,4.168582e+06


## Cross-check against `malata_tau_geometry` — placement closure

Same corpus rows as `malata_tau_geometry` §9/§9b (corpus positions match
`tau_wholesky_n2M_83k.h5` exactly). 277/300 have a forward axis crossing the footprint,
passing a median ~31 m from a grid detector.

They used to trigger only ~2%: `place_clouds_enu` read the native cols 0/1 as C8
transverse coordinates when they are horizontal (East, North) offsets already carrying
the longitudinal development, so every shower was developed twice and swung kilometres
off-axis. Fixed in `modules/dataset_builder.py`.

Below is the closure test — the placed cloud must lie **on** the tau axis (black on red),
and the trigger rate should read ~91%.


In [3]:
# Placement closure: does the placed cloud actually lie on the tau axis?
from modules.data import cloud_to_enu
from plots.lib import geometry_plots

mE_np, mN_np, mU_np = (np.asarray(_mE), np.asarray(_mN), np.asarray(_mU))
P_dec = positions.numpy()
D_dir = dirs_u.numpy()
L_ROD = (N_PLANES - 1) * LAYER_EAST_DX

# Does the axis cross the mountain footprint?
_t = np.linspace(0.0, L_ROD, 600)
_q = P_dec[:, None, :] + _t[None, :, None] * D_dir[:, None, :]
axis_crosses = ((_q[..., 0] >= mE_np.min()) & (_q[..., 0] <= mE_np.max()) &
                (_q[..., 1] >= mN_np.min()) & (_q[..., 1] <= mN_np.max())).any(axis=1)

# Perpendicular distance from axis to nearest detector
_up_scan = surface(n_scan, e_scan).detach().cpu().numpy()
DET = np.stack([e_scan.cpu().numpy(), n_scan.cpu().numpy(), _up_scan], axis=1)
_v = DET[None, :, :] - P_dec[:, None, :]
_s = np.einsum('ijk,ik->ij', _v, D_dir)
_perp = np.linalg.norm(_v - np.clip(_s, 0.0, L_ROD)[:, :, None] * D_dir[:, None, :], axis=2)
axis_dist = _perp.min(axis=1)

df_scan['axis_crosses'] = axis_crosses
df_scan['axis_dist_m'] = axis_dist

print(f"axis crosses footprint: {int(axis_crosses.sum())}/{len(df_scan)}")
print(f"axis->det [m]: p25={np.percentile(axis_dist,25):.0f} "
      f"p50={np.percentile(axis_dist,50):.0f} p90={np.percentile(axis_dist,90):.0f} "
      f"(within sigma: {100*(axis_dist < SIGMA_SPATIAL).mean():.0f}%)")

_near = axis_dist < SIGMA_SPATIAL
print(f"of {int(_near.sum())} near-axis showers: "
      f"{100*(df_scan.e_sum[_near] > 0).mean():.0f}% deposit, "
      f"{100*(df_scan.n_trig[_near] > 0).mean():.0f}% trigger")

# Closure: placed-point distance from own tau axis
_lay_native = torch.as_tensor(
    showerdata.load(DUAL_SHOWER_CACHE_PATH, start=sl.start, stop=sl.stop).points,
    dtype=torch.float32)
_m_live = _lay_native[:, :, 3] > 0
_r_nat = torch.hypot(_lay_native[:, :, 0], _lay_native[:, :, 1])[_m_live].numpy()
print(f"native radius: p50={np.percentile(_r_nat,50):.0f}m p99={np.percentile(_r_nat,99):.0f}m")

_dev = []
for s in np.where(axis_crosses)[0][:60]:
    Q = cloud_to_enu(points[s]) - P_dec[s]
    t = Q @ D_dir[s]
    _dev.append(np.median(np.linalg.norm(Q - t[:, None] * D_dir[s][None, :], axis=1)))
print(f"placed-point median axis distance: {np.median(_dev):.0f} m")

fig = geometry_plots.plot_placement_closure(
    mE_np, mN_np, mU_np, DET, P_dec, D_dir, L_ROD,
    points, energs, axis_dist, df_scan, axis_crosses, cloud_to_enu)
with plt.rc_context({'savefig.bbox': 'tight', 'savefig.pad_inches': 0.3}):
    plt.show()

ImportError: cannot import name 'geometry_plots' from 'plots' (/n/home05/zdimitrov/tambo/TambOpt/plots/__init__.py)

In [ ]:
df_scan

In [ ]:
# Pick 6 showers spanning E and theta, from those that actually deposit something.
# Screening on `n_trig > 0` would leave ~5% of the sample (often zero showers), so
# select on e_sum > 0 — the shower reaches the array — and report n_trig alongside.
df_hit = df_scan[df_scan['e_sum'] > 0].copy()
print(f"{len(df_hit)}/{len(df_scan)} showers deposit a nonzero response; "
      f"{int((df_hit.n_trig > 0).sum())} of those clear LAYOUT_THRESHOLD")
if df_hit.empty:
    raise RuntimeError("no shower reached the array — raise N_SHOWERS_NB in the setup cell")

df_hit['logE'] = np.log10(df_hit['E'])
# q=3/q=2 need enough distinct values; fall back to fewer bins on a thin sample.
_qE  = min(3, df_hit['logE'].nunique())
_qth = min(2, df_hit['theta_deg'].nunique())
df_hit['E_bin']     = pd.qcut(df_hit['logE'], q=_qE,
                              labels=['Elow', 'Emid', 'Ehigh'][:_qE], duplicates='drop')
df_hit['theta_bin'] = pd.qcut(df_hit['theta_deg'], q=_qth,
                              labels=['vert', 'horiz'][:_qth], duplicates='drop')

rng_pick = np.random.default_rng(7)
selected = []
for E_b in df_hit['E_bin'].cat.categories:
    for th_b in df_hit['theta_bin'].cat.categories:
        sub = df_hit[(df_hit['E_bin'] == E_b) & (df_hit['theta_bin'] == th_b)]
        if not len(sub):
            continue
        # Prefer the strongest depositor in the cell so the pattern plots show signal.
        pick = sub.sort_values('e_sum', ascending=False).iloc[0]
        # phi is re-listed explicitly here (like every other field): df_show is built
        # from these dicts, not sliced from df_scan, so anything left out is missing
        # downstream even though df_scan has it.
        selected.append(dict(shower=int(pick['shower']), E=float(pick['E']),
                             theta_deg=float(pick['theta_deg']),
                             phi_deg=float(pick['phi_deg']),
                             n_trig_default=int(pick['n_trig']),
                             e_sum=float(pick['e_sum']),
                             tag=f'{E_b}/{th_b}'))
df_show = pd.DataFrame(selected)
df_show

## For each shower × strategy, reinitialize N_SEEDS times

We regenerate each layout with a fresh RNG so jitter / ring rotation differ between samples. Trigger count is the number of detectors with E > `LAYOUT_THRESHOLD` (imported from `opt_core`, currently 1.0 = "detector saw at least one particle"; it matches the optimization's reconstructability gate).

In [ ]:
N_SEEDS = 15  # reinitializations per (shower, strategy)
records = []
layouts_cache = {}  # (shower, strategy, seed) -> (E (n_det,), e_det, n_det_xy)  [East, North]
layouts_cache_time = {}  # same key -> (E, T, e_det, n_det_xy) -- see cell below

for _, row in df_show.iterrows():
    s = int(row['shower'])
    cloud = points[s:s+1].to(DEVICE)
    for s_idx, (s_name, fn_name, kwargs) in enumerate(_STRATEGIES):
        fn = _STRATEGY_FNS[fn_name]
        for seed in range(N_SEEDS):
            rng = np.random.default_rng(1000 * s_idx + seed)
            e_det, n_det_xy = fn(mountain, n_det=N_DETECTORS, rng=rng, **kwargs)  # (East, North)
            e_det = e_det.float().to(DEVICE); n_det_xy = n_det_xy.float().to(DEVICE)
            # NE kernel defaults are EAST_ENTRY / LAYER_EAST_DX / SIGMA_SPATIAL
            # from constants.py — exactly what Step 1 uses.
            E, T = compute_labels_batch(cloud, e_det, n_det_xy, surface)
            n_trig = int((E[0] > LAYOUT_THRESHOLD).sum().item())
            e_sum  = float(E.sum().item())
            records.append(dict(shower=s, tag=row['tag'], strategy=s_name,
                                seed=seed, n_trig=n_trig, e_sum=e_sum,
                                E_prim=row['E'], theta_deg=row['theta_deg'],
                                phi_deg=row['phi_deg']))
            layouts_cache[(s, s_name, seed)] = (E[0].cpu().numpy(),
                                                e_det.cpu().numpy(),
                                                n_det_xy.cpu().numpy())
            # T: kernel leading-edge arrival time, raw seconds, 0 = no hit
            # (modules/showers/kernel.py). Kept in its own cache rather than
            # widening layouts_cache's tuple, so 05_paper_figures.py's
            # load_trigger_context (its own independent 3-tuple cache) is
            # unaffected.
            layouts_cache_time[(s, s_name, seed)] = (E[0].cpu().numpy(),
                                                     T[0].cpu().numpy(),
                                                     e_det.cpu().numpy(),
                                                     n_det_xy.cpu().numpy())
df = pd.DataFrame(records)
print(f'samples: {len(df)}  ({len(df_show)} showers x {len(_STRATEGIES)} strategies x {N_SEEDS} seeds)')
df.head()

## Bar chart: triggered detectors per (shower, strategy)

Each bar = mean over the `N_SEEDS` reinitializations of a strategy; whiskers = std. Adjacent strategies for the same shower share an x cluster.

In [ ]:
agg = df.groupby(['shower', 'tag', 'strategy'])['n_trig'].agg(['mean', 'std', 'min', 'max']).reset_index()

fig, ax = plt.subplots(1, 1, figsize=(13, 5))
showers = list(df_show['shower'])
tags = {int(r['shower']): r['tag'] for _, r in df_show.iterrows()}
strategy_names = [name for name, _, _ in _STRATEGIES]
n_str = len(strategy_names)
bar_w = 0.15
colors = plt.cm.tab10.colors[:n_str]

for i, s_name in enumerate(strategy_names):
    sub = agg[agg['strategy'] == s_name].set_index('shower').loc[showers]
    xs = np.arange(len(showers)) + (i - (n_str - 1) / 2.0) * bar_w
    ax.bar(xs, sub['mean'], width=bar_w, yerr=sub['std'], label=s_name,
           color=colors[i], capsize=3, alpha=0.85, edgecolor='black', linewidth=0.5)

ax.set_xticks(np.arange(len(showers)))
ax.set_xticklabels([f"#{s}\n{tags[s]}\nE={r['E']:.1e}\nθ={r['theta_deg']:.0f}°  φ={r['phi_deg']:.0f}°"
                    for s, (_, r) in zip(showers, df_show.iterrows())])
ax.set_ylabel(f'# detectors with E > {LAYOUT_THRESHOLD:g} (mean ± std over {N_SEEDS} seeds)')
ax.set_title('Triggered-detector count per (shower, strategy)')
ax.legend(loc='upper right', ncol=2, fontsize=9)
ax.grid(alpha=0.3, axis='y')
fig.tight_layout(); plt.show()

## Detector patterns for representative (shower, layout) pairs

For each of the 6 showers, pick one strategy seed with mid-range trigger count and plot the layout in the **same 3D ENU view as the placement-closure figure above**, so the shower is visible relative to the array instead of flattened into the cliff plane: mountain in `#DDD6C9`, detectors as triangles (grey where dark, `plasma` by raw counts where lit, cyan-ringed above `LAYOUT_THRESHOLD`), the placed cloud in green, and the tau axis in red from its decay vertex.

The view is clipped to `VIEW_R` metres around the array. The rod is ~11.5 km long (`L_ROD`) and the array only 1.4 km, so autoscaling to the whole shower — what the closure figure does — shrinks the detector pattern to a smudge. Set `VIEW_R = None` to see the entire rod and its decay vertex instead; the tau axis is clipped to the box either way, so the incoming direction (the title's φ) still reads.

In [ ]:
import importlib.util as _ilu
from matplotlib.ticker import MaxNLocator
from modules.data import cloud_to_enu
from plots.lib import geometry_plots

_spec = _ilu.spec_from_file_location("opt_plotting",
                                     os.path.join(_V6, "plots", "lib", "opt_plotting.py"))
_optplt = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_optplt)

MTN_ENU  = _optplt.mountain_enu(mountain)   # (n, 3) East, North, Up
SURF_CPU = surface.to("cpu")                # projection is a read-only plotting step

# Half-width [m] of the East/North box drawn around the array centre. None =
# autoscale to the whole 11.5 km rod (the closure figure's framing), which makes
# the 1.4 km array unreadable — the reason this exists.
VIEW_R = 1500.0
# Denser and larger than the closure figure's ::8 / s=2: that was tuned to stop a
# 25k-point rod burying the detectors, but the clip already removes most of it.
CLOUD_STRIDE, CLOUD_S, CLOUD_A = 2, 8, 0.85

CTR = MTN_ENU.mean(axis=0)
UP_LO, UP_HI = MTN_ENU[:, 2].min() - 300.0, MTN_ENU[:, 2].max() + 300.0

def _in_view(P):
    """Mask of ENU points inside the drawn box (all True when VIEW_R is None)."""
    if VIEW_R is None:
        return np.ones(len(P), dtype=bool)
    return ((np.abs(P[:, 0] - CTR[0]) <= VIEW_R)
            & (np.abs(P[:, 1] - CTR[1]) <= VIEW_R)
            & (P[:, 2] >= UP_LO) & (P[:, 2] <= UP_HI))

def panel(ax, E_arr, e_det, n_det_xy, cloud=None, s=None):
    """One 3D ENU panel: mountain + placed cloud + tau axis + detectors by count.

    Same palette and viewing angle as the placement-closure figure, but detectors
    carry the count colour scale (`draw_detectors_enu_3d`) and the scene is clipped
    to VIEW_R. Passing `s` (a shower index) adds that shower's tau axis, and its
    decay vertex when the vertex falls inside the box.

    Kept here (not moved into geometry_plots) because "same shower, different
    reinitializations" below calls this exact closure again.
    """

    ax.scatter(MTN_ENU[:, 0], MTN_ENU[:, 1], MTN_ENU[:, 2], s=3, c="#DDD6C9",
               alpha=0.6, depthshade=False, label="mountain")
    if cloud is not None:
        P_cloud = cloud_to_enu(cloud)
        P_cloud = P_cloud[_in_view(P_cloud)][::CLOUD_STRIDE]
        if len(P_cloud):
            ax.scatter(P_cloud[:, 0], P_cloud[:, 1], P_cloud[:, 2], s=CLOUD_S,
                       c="tab:green", alpha=CLOUD_A, linewidths=0, depthshade=False,
                       label="shower cloud")
    if s is not None:
        # P_dec / D_dir / L_ROD come from the placement-closure cell above. Sampled
        # finely, not as two endpoints, so clipping leaves the in-box segment rather
        # than dropping the whole line.
        axis = (P_dec[s][None, :]
                + np.linspace(0, L_ROD, 4000)[:, None] * D_dir[s][None, :])
        seg = axis[_in_view(axis)]
        # if len(seg):
        #     ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], c="red", lw=1.6,
        #             label="tau axis")
        if _in_view(P_dec[s][None, :])[0]:
            ax.scatter(*P_dec[s], c="red", s=110, marker="*", depthshade=False,
                       label="decay vertex")

    sc, n_live, n_over = _optplt.draw_detectors_enu_3d(
        ax, E_arr, e_det, n_det_xy, SURF_CPU,
        layout_threshold=LAYOUT_THRESHOLD, log1p_space=False)

    if VIEW_R is not None:
        ax.set_xlim(CTR[0] - VIEW_R, CTR[0] + VIEW_R)
        ax.set_ylim(CTR[1] - VIEW_R, CTR[1] + VIEW_R)
        ax.set_zlim(UP_LO, UP_HI)
    ax.set_xlabel("East [m]", labelpad=8)
    ax.set_ylabel("North [m]", labelpad=8)
    ax.set_zlabel("Up [m]", labelpad=2)
    ax.tick_params(axis="x", pad=2); ax.tick_params(axis="y", pad=2)
    ax.tick_params(axis="z", pad=0)
    for a in (ax.xaxis, ax.yaxis, ax.zaxis):
        a.set_major_locator(MaxNLocator(nbins=4))
    ax.view_init(elev=25, azim=-125)
    return sc, n_live, n_over

EXAMPLE_STRATEGY = 'center_gauss400'
fig = geometry_plots.plot_detector_patterns(
    panel, df, df_show, layouts_cache, points, EXAMPLE_STRATEGY, LAYOUT_THRESHOLD)
with plt.rc_context({'savefig.bbox': 'tight', 'savefig.pad_inches': 0.3}):
    plt.show()

## Detector patterns, coloured by arrival time

Same six representative (shower, layout) picks and the same 3D ENU view as the figure above, but detectors are coloured by **arrival time** `T` instead of **counts** `E`. `T` is the kernel's leading-edge arrival time — the earliest detected particle's time in raw seconds, `0` = no hit (`modules/showers/kernel.py:GetCounts_planeaware`), not a weighted mean — so it is the observable an analytic front-timing fit (plane χ² or a front-curvature term) would actually use.

The dark/lit split is unchanged (still `E`-derived, so both figures show the identical set of lit detectors); only the colour channel on the lit set differs, and it is drawn on a **linear** scale (µs) rather than `E`'s log-count scale, since the front's timing structure — not its dynamic range — is the point.


In [ ]:
# Same 3D `panel` machinery as the E-coloured figure above, but colouring the
# lit detectors by arrival time (`draw_detectors_enu_3d_by_time`) instead of
# counts. Dark/lit is still decided from E_arr, so this shows exactly the same
# lit set as the previous figure -- only the colour channel changes.
def panel_time(ax, E_arr, T_arr, e_det, n_det_xy, cloud=None, s=None):
    """Twin of `panel` above, coloured by arrival time instead of counts.

    Kept here (not moved into geometry_plots) for the same reason as `panel`:
    a later cell could reuse this exact closure on a different seed set.
    """
    ax.scatter(MTN_ENU[:, 0], MTN_ENU[:, 1], MTN_ENU[:, 2], s=3, c="#DDD6C9",
               alpha=0.6, depthshade=False, label="mountain")
    if cloud is not None:
        P_cloud = cloud_to_enu(cloud)
        P_cloud = P_cloud[_in_view(P_cloud)][::CLOUD_STRIDE]
        if len(P_cloud):
            ax.scatter(P_cloud[:, 0], P_cloud[:, 1], P_cloud[:, 2], s=CLOUD_S,
                       c="tab:green", alpha=CLOUD_A, linewidths=0, depthshade=False,
                       label="shower cloud")
    if s is not None:
        # Same convention as `panel`: only the decay vertex marker (not the
        # tau-axis line, commented out there too) is drawn.
        if _in_view(P_dec[s][None, :])[0]:
            ax.scatter(*P_dec[s], c="red", s=110, marker="*", depthshade=False,
                       label="decay vertex")

    sc, n_live, n_over = _optplt.draw_detectors_enu_3d_by_time(
        ax, E_arr, T_arr, e_det, n_det_xy, SURF_CPU,
        layout_threshold=LAYOUT_THRESHOLD)

    if VIEW_R is not None:
        ax.set_xlim(CTR[0] - VIEW_R, CTR[0] + VIEW_R)
        ax.set_ylim(CTR[1] - VIEW_R, CTR[1] + VIEW_R)
        ax.set_zlim(UP_LO, UP_HI)
    ax.set_xlabel("East [m]", labelpad=8)
    ax.set_ylabel("North [m]", labelpad=8)
    ax.set_zlabel("Up [m]", labelpad=2)
    ax.tick_params(axis="x", pad=2); ax.tick_params(axis="y", pad=2)
    ax.tick_params(axis="z", pad=0)
    for a in (ax.xaxis, ax.yaxis, ax.zaxis):
        a.set_major_locator(MaxNLocator(nbins=4))
    ax.view_init(elev=25, azim=-125)
    return sc, n_live, n_over

fig = geometry_plots.plot_detector_patterns_by_time(
    panel_time, df, df_show, layouts_cache_time, points, EXAMPLE_STRATEGY, LAYOUT_THRESHOLD)
with plt.rc_context({'savefig.bbox': 'tight', 'savefig.pad_inches': 0.3}):
    plt.show()


## Same shower, different layout reinitializations

Pin one shower (highest mean trigger count) and one strategy, then show 6 different reinitializations side by side. This isolates the variance from layout randomness alone.

In [ ]:
# Same shower, several layout reinitializations — same 3D `panel` helper.
best_row = (agg[agg['strategy'] == EXAMPLE_STRATEGY]
            .sort_values('mean', ascending=False).iloc[0])
s = int(best_row['shower'])
tag = tags[s]
phis = {int(r['shower']): float(r['phi_deg']) for _, r in df_show.iterrows()}
thetas = {int(r['shower']): float(r['theta_deg']) for _, r in df_show.iterrows()}
n_show = min(6, N_SEEDS)          # never index past the seeds actually generated
fig, axes = plt.subplots(2, 3, figsize=(19, 11), subplot_kw=dict(projection='3d'))
for ax in axes.flat[n_show:]:
    ax.set_visible(False)

for ax, seed in zip(axes.flat, range(n_show)):
    E_arr, e_det, n_det_xy = layouts_cache[(s, EXAMPLE_STRATEGY, seed)]
    sc, n_live, n_over = panel(ax, E_arr, e_det, n_det_xy, cloud=points[s], s=s)
    if sc is not None:
        plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.10, label='counts (raw)')
    ax.set_title(f'seed={seed}  signal={n_live}/100  >{LAYOUT_THRESHOLD:g}={n_over}', fontsize=9)

fig.suptitle(f"Shower #{s} ({tag}, θ={thetas[s]:.0f}°, φ={phis[s]:.0f}°), "
             f"strategy={EXAMPLE_STRATEGY} — "
             f"{n_show} layout reinitializations in 3D ENU", fontsize=12)
fig.tight_layout(pad=1.4)
with plt.rc_context({'savefig.bbox': 'tight', 'savefig.pad_inches': 0.3}):
    plt.show()

## Distribution of trigger counts across all (shower, layout) pairs sampled here

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['n_trig'], bins=range(0, df['n_trig'].max() + 3), edgecolor='black')
axes[0].set_xlabel('# triggered detectors'); axes[0].set_ylabel('# (shower, layout) pairs')
axes[0].set_title(f'Distribution across {len(df)} sampled pairs')
axes[0].grid(alpha=0.3)

for s_name in [n for n, _, _ in _STRATEGIES]:
    axes[1].hist(df[df['strategy']==s_name]['n_trig'], bins=range(0, df['n_trig'].max() + 3),
                 histtype='step', linewidth=1.5, label=s_name)
axes[1].set_xlabel('# triggered detectors'); axes[1].set_ylabel('count')
axes[1].set_title('Per-strategy distribution'); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

## Summary table

In [ ]:
summary = df.groupby(['tag', 'strategy'])['n_trig'].agg(['mean', 'std', 'min', 'max']).round(2)
summary